# Otimização de rotas hospitalares

Este notebook reproduz o planejamento do cenário fictício. A análise começa por uma solução de referência, executa o algoritmo genético e examina viabilidade, ganho relativo e convergência.

In [ ]:
from pathlib import Path
import time
import sys
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
from hospital_routes.io import load_problem
from hospital_routes.genetic import GAConfig, GeneticOptimizer
from hospital_routes.baselines import brute_force, nearest_neighbor
from hospital_routes.reporting import (GeminiReportGenerator, LocalReportGenerator, comparison_metrics, configured_gemini_key)
from hospital_routes.visualization import save_route_map, save_convergence_plot
print(f'Kernel: {sys.executable}')
print('Gemini configurado:', bool(configured_gemini_key()))

## Cenário e configuração

A semente fixa torna a execução repetível. As penalidades de capacidade e autonomia são deliberadamente maiores que o custo de alguns quilômetros adicionais.

In [ ]:
problem = load_problem(ROOT / 'data' / 'deliveries.json')
config = GAConfig(population_size=120, generations=300, seed=42)
optimizer = GeneticOptimizer(problem, config)
baseline = nearest_neighbor(optimizer)
solution = optimizer.run()
{'entregas': len(problem.deliveries), 'veiculos': len(problem.vehicles), 'geracoes_executadas': len(optimizer.history)}

### Diagnóstico prévio de viabilidade

A soma das capacidades precisa ser suficiente para a demanda total, mas essa é apenas uma condição necessária: cada rota ainda deve respeitar individualmente carga e autonomia. A verificação prévia ajuda a distinguir dificuldade de busca de inviabilidade estrutural.

In [ ]:
total_demand = sum(item.demand_kg for item in problem.deliveries)
total_capacity = sum(vehicle.capacity_kg for vehicle in problem.vehicles)
largest_demand = max(item.demand_kg for item in problem.deliveries)
largest_capacity = max(vehicle.capacity_kg for vehicle in problem.vehicles)
print(f'Demanda total: {total_demand:.1f} kg')
print(f'Capacidade total: {total_capacity:.1f} kg')
print('Capacidade agregada suficiente:', total_demand <= total_capacity)
print('Maior entrega cabe em algum veículo:', largest_demand <= largest_capacity)

## Resultado comparativo

A comparação utiliza a mesma aptidão. Percentual positivo indica redução do custo composto em relação ao vizinho mais próximo.

In [ ]:
gain = 100 * (baseline.fitness - solution.fitness) / baseline.fitness
print(f'Fitness baseline: {baseline.fitness:.2f}')
print(f'Fitness GA: {solution.fitness:.2f}')
print(f'Redução: {gain:.2f}% | Distância: {solution.total_distance_km:.2f} km | Viável: {solution.feasible}')
[(problem.vehicles[i].id, [problem.deliveries[j].id for j in route]) for i, route in enumerate(solution.routes)]

### Interpretação do comparativo

O fitness não representa somente quilômetros: ele incorpora atraso ponderado por prioridade e penalidades por violações. Por isso, uma redução de fitness não deve ser apresentada automaticamente como a mesma redução de distância, tempo ou custo financeiro.

In [ ]:
comparison = comparison_metrics(solution, baseline)
for key, value in comparison.items():
    print(f'{key}: {value}')

## Visualizações

O gráfico permite verificar estabilização da população. O mapa HTML conserva a ordem das visitas e o retorno ao depósito.

In [ ]:
outputs = ROOT / 'outputs'
outputs.mkdir(exist_ok=True)
save_convergence_plot(optimizer.history, outputs / 'convergence.png')
save_route_map(problem, solution, outputs / 'routes_map.html')
from IPython.display import Image, display
display(Image(filename=outputs / 'convergence.png'))

### Leitura dos gráficos

O melhor fitness indica a qualidade do indivíduo de elite, enquanto a média descreve a população. A queda de ambos sugere evolução coletiva; estabilização prolongada pode indicar convergência. Os painéis usam escalas independentes para preservar a leitura de cada série.

## Instruções operacionais

A solução estruturada é enviada ao modelo Gemini configurado no projeto para produzir as instruções operacionais.

In [ ]:
try:
    report = GeminiReportGenerator().generate(problem, solution, comparison=comparison)
    print('Relatório gerado pelo Gemini.')
except RuntimeError as error:
    report = LocalReportGenerator().generate(problem, solution, comparison=comparison)
    print(f'Gemini indisponível; relatório local utilizado. Motivo: {error}')
print(report)

### Fundamentação e limites da geração textual

A LLM recebe a solução após a otimização e não participa da escolha das rotas. O prompt inclui papel, tarefa, contexto JSON e proibição de inventar tempo ou economia monetária. Mesmo assim, a saída requer revisão humana devido ao risco de alucinação.